# Model A: Custom CNN

Binary classification of real and AI-generated images
using a Custom Convolutional Neural Network.

Training environment: Google Colab
Dataset: Defactify Image Dataset
Framework: TensorFlow / Keras

In [25]:
!pip -q install datasets huggingface_hub

In [26]:
import os
import json
import shutil
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from pathlib import Path
from PIL import Image as PILImage
from datasets import load_dataset
from huggingface_hub import HfApi

In [27]:
# GitHub repository
REPO_URL = (
    "https://github.com/Michaelatj/"
    "AI-Generated-Image-Classification-using-CNN.git"
)

REPO_DIR = Path(
    "/content/AI-Generated-Image-Classification-using-CNN"
)

# Clone repository if not available
if not REPO_DIR.exists():

    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        check=True
    )

os.chdir(REPO_DIR)

# Sync with latest GitHub main branch
status = subprocess.check_output(
    ["git", "status", "--porcelain"],
    text=True
)

if status.strip():
    raise RuntimeError(
        "Repository contains local changes. "
        "Check git status before syncing."
    )

subprocess.run(
    ["git", "pull", "--rebase", "origin", "main"],
    check=True
)

# Project directories
ARTIFACT_DIR = REPO_DIR / "artifacts"
SPLIT_DIR = ARTIFACT_DIR / "splits"
CONFIG_DIR = ARTIFACT_DIR / "configs"
MODEL_DIR = ARTIFACT_DIR / "models"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Repository:", REPO_DIR)
print("Models directory:", MODEL_DIR)

Repository: /content/AI-Generated-Image-Classification-using-CNN
Models directory: /content/AI-Generated-Image-Classification-using-CNN/artifacts/models


In [28]:
# TensorFlow configuration
SEED = 42

tf.keras.utils.set_random_seed(SEED)

# GPU configuration
gpus = tf.config.list_physical_devices("GPU")

for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(
            gpu,
            True
        )
    except RuntimeError as error:
        print(error)

print("TensorFlow:", tf.__version__)
print("Available GPU:", gpus)

# Check available Colab storage
total, used, free = shutil.disk_usage("/content")

print(f"Free storage: {free / (1024**3):.2f} GB")

TensorFlow: 2.21.0
Available GPU: []
Free storage: 107.59 GB


## Dataset Loading

Load the prepared CSV manifests and original images
from the Hugging Face Defactify Image Dataset.

Target:
- 0 = Real
- 1 = AI-Generated

The test split is reserved for final evaluation.

In [29]:
TRAIN_CSV = SPLIT_DIR / "train.csv"
VAL_CSV = SPLIT_DIR / "validation.csv"
TEST_CSV = SPLIT_DIR / "test.csv"

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

# Sort to make image retrieval and prediction ordering consistent
train_df = train_df.sort_values(
    ["source_split", "row_idx"]
).reset_index(drop=True)

val_df = val_df.sort_values(
    ["source_split", "row_idx"]
).reset_index(drop=True)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

display(train_df.head())

Train: (42000, 9)
Validation: (9000, 9)
Test: (45000, 9)


,image_id,source_split,row_idx,caption_group_id,caption,label_a,label_a_name,label_b,label_b_name
0,train_000000,train,0,6dfa76b111164685,Two tall giraffe standing next to each other o...,0,Real,0,Real
1,train_000001,train,1,6dfa76b111164685,Two tall giraffe standing next to each other o...,1,AI-Generated,1,Stable Diffusion 2.1
2,train_000002,train,2,6dfa76b111164685,Two tall giraffe standing next to each other o...,1,AI-Generated,2,Stable Diffusion XL (SDXL)
3,train_000003,train,3,6dfa76b111164685,Two tall giraffe standing next to each other o...,1,AI-Generated,3,Stable Diffusion 3
4,train_000004,train,4,6dfa76b111164685,Two tall giraffe standing next to each other o...,1,AI-Generated,4,DALL-E 3


In [30]:
DATASET_ID = (
    "Rajarshi-Roy-research/Defactify_Image_Dataset"
)

HF_CACHE_DIR = "/content/huggingface_cache"

# Pin the dataset revision for reproducibility
HF_REVISION = HfApi().dataset_info(
    DATASET_ID
).sha

print("Dataset revision:", HF_REVISION)

HF_DATASETS = {}

# Only load the splits needed for training
required_splits = sorted(
    set(train_df["source_split"]).union(
        set(val_df["source_split"])
    )
)

for split_name in required_splits:

    print(f"\nLoading {split_name}...")

    HF_DATASETS[split_name] = load_dataset(
        DATASET_ID,
        split=split_name,
        revision=HF_REVISION,
        cache_dir=HF_CACHE_DIR
    )

    print(
        f"{split_name}: "
        f"{len(HF_DATASETS[split_name]):,} images"
    )

Dataset revision: 1c3b714c9691eb6dd11c2c1e92eab1ac29584fa0

Loading train...


data/validation-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  333MB            

data/validation-00000-of-00002.parquet: downloading bytes:           |  0.00B            

data/validation-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  345MB            

data/validation-00001-of-00002.parquet: downloading bytes:           |  0.00B            

data/train-00000-of-00007.parquet: reconstructing file:   0%|          |  0.00B /  448MB            

... (29 outputs left out)

Generating validation split:   0%|          | 0/9000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/42000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/45000 [00:00<?, ? examples/s]

train: 42,000 images

Loading validation...
validation: 9,000 images


In [32]:
# Find overlapping caption groups
overlap_groups = (
    set(train_df["caption_group_id"].dropna())
    &
    set(val_df["caption_group_id"].dropna())
)

print("Total overlapping groups:", len(overlap_groups))

# Inspect overlapping groups
overlap_details = []

for group_id in sorted(overlap_groups):

    train_group = train_df[
        train_df["caption_group_id"] == group_id
    ]

    val_group = val_df[
        val_df["caption_group_id"] == group_id
    ]

    overlap_details.append({

        "caption_group_id": group_id,

        "train_rows": len(train_group),

        "validation_rows": len(val_group),

        "train_images": train_group["image_id"].tolist(),

        "validation_images": val_group["image_id"].tolist(),

        "caption": train_group["caption"].iloc[0]

    })

overlap_report = pd.DataFrame(overlap_details)

display(overlap_report)

Total overlapping groups: 26


,caption_group_id,train_rows,validation_rows,train_images,validation_images,caption
0,04152f72e03e84f9,6,6,"[train_036690, train_036691, train_036692, tra...","[validation_000706, validation_003204, validat...",Two giraffe standing next to each other on a l...
1,08ca47e762cbbccb,6,6,"[train_018840, train_018841, train_018842, tra...","[validation_000082, validation_002448, validat...",A plane that is flying in the air.
2,0c26fef439c101d2,6,6,"[train_023736, train_023737, train_023738, tra...","[validation_002757, validation_005102, validat...",A zebra and giraffe that is in the dirt.
3,37a02b98ce3860eb,12,6,"[train_020976, train_020977, train_020978, tra...","[validation_000239, validation_003630, validat...",a train on a train track at a train station
4,3cbeadf2da578708,6,6,"[train_001614, train_001615, train_001616, tra...","[validation_001374, validation_002130, validat...",A clock that is on the side of a tower.
5,3ee470dda0408488,6,6,"[train_041268, train_041269, train_041270, tra...","[validation_001347, validation_003810, validat...",A airplane that is flying in the sky.
6,42145dca02d3adaf,6,6,"[train_006492, train_006493, train_006494, tra...","[validation_001133, validation_001536, validat...",a bathroom with a toilet and a bath tub
7,4b4afe85715b193e,12,6,"[train_002172, train_002173, train_002174, tra...","[validation_000648, validation_002901, validat...",A red double decker bus driving down a street.
8,505c642286e4bb79,6,6,"[train_035238, train_035239, train_035240, tra...","[validation_000715, validation_002157, validat...",a public transit bus on a city street
9,70e0da06e26ff9bd,12,6,"[train_001560, train_001561, train_001562, tra...","[validation_000077, validation_003925, validat...",A man riding on the back of a motorcycle down ...


In [33]:
# Display first 3 overlapping groups

for group_id in sorted(overlap_groups)[:3]:

    print(f"\nCaption Group: {group_id}")

    train_group = train_df[
        train_df["caption_group_id"] == group_id
    ].copy()

    val_group = val_df[
        val_df["caption_group_id"] == group_id
    ].copy()

    train_group["dataset_split"] = "Train"
    val_group["dataset_split"] = "Validation"

    combined = pd.concat([
        train_group,
        val_group
    ])

    display(
        combined[[
            "image_id",
            "dataset_split",
            "caption_group_id",
            "label_a",
            "label_b",
            "caption"
        ]]
    )


Caption Group: 04152f72e03e84f9


,image_id,dataset_split,caption_group_id,label_a,label_b,caption
36690,train_036690,Train,04152f72e03e84f9,0,0,Two giraffe standing next to each other on a l...
36691,train_036691,Train,04152f72e03e84f9,1,1,Two giraffe standing next to each other on a l...
36692,train_036692,Train,04152f72e03e84f9,1,2,Two giraffe standing next to each other on a l...
36693,train_036693,Train,04152f72e03e84f9,1,3,Two giraffe standing next to each other on a l...
36694,train_036694,Train,04152f72e03e84f9,1,4,Two giraffe standing next to each other on a l...
36695,train_036695,Train,04152f72e03e84f9,1,5,Two giraffe standing next to each other on a l...
706,validation_000706,Validation,04152f72e03e84f9,1,5,Two giraffe standing next to each other on a l...
3204,validation_003204,Validation,04152f72e03e84f9,1,1,Two giraffe standing next to each other on a l...
3545,validation_003545,Validation,04152f72e03e84f9,1,2,Two giraffe standing next to each other on a l...
4597,validation_004597,Validation,04152f72e03e84f9,1,3,Two giraffe standing next to each other on a l...



Caption Group: 08ca47e762cbbccb


,image_id,dataset_split,caption_group_id,label_a,label_b,caption
18840,train_018840,Train,08ca47e762cbbccb,0,0,A plane that is flying in the air.
18841,train_018841,Train,08ca47e762cbbccb,1,1,A plane that is flying in the air.
18842,train_018842,Train,08ca47e762cbbccb,1,2,A plane that is flying in the air.
18843,train_018843,Train,08ca47e762cbbccb,1,3,A plane that is flying in the air.
18844,train_018844,Train,08ca47e762cbbccb,1,4,A plane that is flying in the air.
18845,train_018845,Train,08ca47e762cbbccb,1,5,A plane that is flying in the air.
82,validation_000082,Validation,08ca47e762cbbccb,1,1,A plane that is flying in the air.
2448,validation_002448,Validation,08ca47e762cbbccb,1,5,A plane that is flying in the air.
2630,validation_002630,Validation,08ca47e762cbbccb,1,4,A plane that is flying in the air.
3523,validation_003523,Validation,08ca47e762cbbccb,1,3,A plane that is flying in the air.



Caption Group: 0c26fef439c101d2


,image_id,dataset_split,caption_group_id,label_a,label_b,caption
23736,train_023736,Train,0c26fef439c101d2,0,0,A zebra and giraffe that is in the dirt.
23737,train_023737,Train,0c26fef439c101d2,1,1,A zebra and giraffe that is in the dirt.
23738,train_023738,Train,0c26fef439c101d2,1,2,A zebra and giraffe that is in the dirt.
23739,train_023739,Train,0c26fef439c101d2,1,3,A zebra and giraffe that is in the dirt.
23740,train_023740,Train,0c26fef439c101d2,1,4,A zebra and giraffe that is in the dirt.
23741,train_023741,Train,0c26fef439c101d2,1,5,A zebra and giraffe that is in the dirt.
2757,validation_002757,Validation,0c26fef439c101d2,1,5,A zebra and giraffe that is in the dirt.
5102,validation_005102,Validation,0c26fef439c101d2,1,3,A zebra and giraffe that is in the dirt.
5876,validation_005876,Validation,0c26fef439c101d2,1,4,A zebra and giraffe that is in the dirt.
6369,validation_006369,Validation,0c26fef439c101d2,1,1,A zebra and giraffe that is in the dirt.


In [34]:
split_groups = {
    "train": set(train_df["caption_group_id"].dropna()),
    "validation": set(val_df["caption_group_id"].dropna()),
    "test": set(test_df["caption_group_id"].dropna())
}

print(
    "Train - Validation:",
    len(split_groups["train"] & split_groups["validation"])
)

print(
    "Train - Test:",
    len(split_groups["train"] & split_groups["test"])
)

print(
    "Validation - Test:",
    len(split_groups["validation"] & split_groups["test"])
)

Train - Validation: 26
Train - Test: 34
Validation - Test: 8


In [31]:
def normalize_caption(text):
    return " ".join(
        str(text).strip().split()
    ).casefold()


required_columns = [
    "image_id",
    "source_split",
    "row_idx",
    "caption",
    "label_a",
    "label_b"
]

for name, df in [
    ("train", train_df),
    ("validation", val_df)
]:

    assert set(required_columns).issubset(df.columns)

    assert df["image_id"].is_unique

    assert not df[
        ["source_split", "row_idx", "caption", "label_a", "label_b"]
    ].isnull().any().any()

    assert df[["source_split", "row_idx"]].duplicated().sum() == 0

    assert set(df["label_a"].unique()).issubset({0, 1})


def validate_mapping(df, split_name):

    for source_split, group in df.groupby(
        "source_split",
        sort=False
    ):

        hf_dataset = HF_DATASETS[source_split]

        indices = group["row_idx"].astype(int).tolist()

        if min(indices) < 0 or max(indices) >= len(hf_dataset):
            raise IndexError(
                f"Invalid row_idx in {split_name}"
            )

        # Select original Hugging Face rows
        # without accessing the image column
        selected = hf_dataset.select(indices)

        actual_labels_a = np.asarray(
            selected["Label_A"]
        )

        actual_labels_b = np.asarray(
            selected["Label_B"]
        )

        expected_labels_a = group["label_a"].to_numpy()
        expected_labels_b = group["label_b"].to_numpy()

        assert np.array_equal(
            actual_labels_a,
            expected_labels_a
        ), f"Label_A mismatch: {split_name}"

        assert np.array_equal(
            actual_labels_b,
            expected_labels_b
        ), f"Label_B mismatch: {split_name}"

        actual_captions = [
            normalize_caption(x)
            for x in selected["Caption"]
        ]

        expected_captions = [
            normalize_caption(x)
            for x in group["caption"]
        ]

        assert actual_captions == expected_captions, (
            f"Caption mismatch: {split_name}"
        )

    print(
        f"{split_name}: image mapping verified!"
    )


validate_mapping(train_df, "train")
validate_mapping(val_df, "validation")

# Check caption group leakage
if "caption_group_id" in train_df.columns:

    train_groups = set(train_df["caption_group_id"].dropna())
    val_groups = set(val_df["caption_group_id"].dropna())

    overlap = train_groups & val_groups

    assert len(overlap) == 0, (
        f"Caption group leakage detected: {len(overlap)} groups"
    )

    print("No caption group leakage detected.")

train: image mapping verified!
validation: image mapping verified!


AssertionError: Caption group leakage detected: 26 groups

In [ ]:
fig, axes = plt.subplots(
    2, 4,
    figsize=(14, 7)
)

sample_rows = train_df.sample(
    n=8,
    random_state=SEED
)

for ax, (_, row) in zip(
    axes.flatten(),
    sample_rows.iterrows()
):

    sample = HF_DATASETS[
        row["source_split"]
    ][int(row["row_idx"])]

    image = sample["Image"].convert("RGB")

    ax.imshow(image)

    label_name = (
        "Real"
        if row["label_a"] == 0
        else "AI-Generated"
    )

    ax.set_title(label_name)
    ax.axis("off")

plt.tight_layout()
plt.show()

## Image Preprocessing and Data Pipeline

Retrieve images using source_split and row_idx.

Preprocessing:
- RGB conversion.
- Resize to 224 × 224.
- Convert to uint8 image tensors.
- Normalize pixel values inside the CNN.

Use tf.data for batching, shuffling, and prefetching.

In [ ]:
IMAGE_SIZE = 224
IMAGE_SHAPE = (224, 224, 3)

BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

# Calculate class weights from training data
class_counts = (
    train_df["label_a"]
    .value_counts()
    .reindex([0, 1], fill_value=0)
)

if (class_counts == 0).any():
    raise ValueError("A training class is missing!")

total_samples = len(train_df)

class_weights = {
    label: total_samples / (
        2 * class_counts[label]
    )
    for label in [0, 1]
}

print("Class distribution:")
print(class_counts)

print("\nClass weights:")
print(class_weights)


def image_generator(df, training=False):

    for row in df.itertuples(index=False):

        # Retrieve original image
        sample = HF_DATASETS[
            row.source_split
        ][int(row.row_idx)]

        image = sample["Image"]

        if image is None:
            raise ValueError(
                f"Missing image: {row.image_id}"
            )

        # Convert to RGB and resize
        image = image.convert("RGB")

        image = image.resize(
            (IMAGE_SIZE, IMAGE_SIZE),
            resample=PILImage.Resampling.BILINEAR
        )

        # Convert to NumPy array
        image_array = np.asarray(
            image,
            dtype=np.uint8
        )

        # Binary label
        label = np.array(
            [int(row.label_a)],
            dtype=np.float32
        )

        # Apply class weighting only to training data
        sample_weight = np.float32(
            class_weights[int(row.label_a)]
            if training else 1.0
        )

        yield image_array, label, sample_weight

In [ ]:
OUTPUT_SIGNATURE = (

    tf.TensorSpec(
        shape=IMAGE_SHAPE,
        dtype=tf.uint8
    ),

    tf.TensorSpec(
        shape=(1,),
        dtype=tf.float32
    ),

    tf.TensorSpec(
        shape=(),
        dtype=tf.float32
    )
)


def create_tf_dataset(df, training=False):

    dataset = tf.data.Dataset.from_generator(
        lambda: image_generator(
            df,
            training=training
        ),
        output_signature=OUTPUT_SIGNATURE
    )

    # Verify expected number of samples
    dataset = dataset.apply(
        tf.data.experimental.assert_cardinality(
            len(df)
        )
    )

    if training:

        dataset = dataset.shuffle(
            buffer_size=512,
            seed=SEED,
            reshuffle_each_iteration=True
        )

    dataset = dataset.batch(
        BATCH_SIZE
    )

    dataset = dataset.prefetch(
        AUTOTUNE
    )

    return dataset


train_dataset = create_tf_dataset(
    train_df,
    training=True
)

val_dataset = create_tf_dataset(
    val_df,
    training=False
)

print("Train and validation pipelines created!")

In [ ]:
images, labels, sample_weights = next(
    iter(train_dataset)
)

print("Image shape:", images.shape)
print("Label shape:", labels.shape)
print("Sample weight shape:", sample_weights.shape)

print("Image dtype:", images.dtype)
print("Label dtype:", labels.dtype)

# Display training samples
plt.figure(figsize=(12, 7))

for i in range(min(8, len(images))):

    plt.subplot(2, 4, i + 1)

    plt.imshow(
        images[i].numpy()
    )

    label = int(
        labels[i][0].numpy()
    )

    plt.title(
        "Real" if label == 0 else "AI-Generated"
    )

    plt.axis("off")

plt.tight_layout()
plt.show()

## Custom CNN Architecture

Build a CNN from scratch for binary image classification.

Architecture:
- Input: 224 × 224 × 3
- Three convolution blocks: 32, 64, 128 filters
- Batch Normalization
- ReLU activation
- Max Pooling
- Global Average Pooling
- Dense: 128 neurons
- Dropout
- Sigmoid output

Training configuration:
- Optimizer: Adam
- Learning rate: 0.001
- Loss: Binary Crossentropy
- Batch size: 16
- Maximum epochs: 10

In [ ]:
from tensorflow.keras import layers, models


def conv_block(x, filters, dropout_rate=0.0):

    # First convolution
    x = layers.Conv2D(
        filters,
        kernel_size=3,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    # Second convolution
    x = layers.Conv2D(
        filters,
        kernel_size=3,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    # Downsampling
    x = layers.MaxPooling2D(
        pool_size=2
    )(x)

    if dropout_rate > 0:
        x = layers.Dropout(dropout_rate)(x)

    return x


def build_custom_cnn():

    inputs = layers.Input(
        shape=IMAGE_SHAPE
    )

    # Normalize image pixels from 0-255 to 0-1
    x = layers.Rescaling(1.0 / 255)(inputs)

    # Convolution blocks
    x = conv_block(x, 32)
    x = conv_block(x, 64, dropout_rate=0.10)
    x = conv_block(x, 128, dropout_rate=0.20)

    # Feature aggregation
    x = layers.GlobalAveragePooling2D()(x)

    # Classification head
    x = layers.Dense(
        128,
        activation="relu"
    )(x)

    x = layers.Dropout(0.4)(x)

    outputs = layers.Dense(
        1,
        activation="sigmoid"
    )(x)

    return models.Model(
        inputs=inputs,
        outputs=outputs,
        name="Custom_CNN"
    )


model = build_custom_cnn()

model.summary()

In [ ]:
model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),

    loss="binary_crossentropy",

    metrics=[

        tf.keras.metrics.BinaryAccuracy(
            name="accuracy"
        ),

        tf.keras.metrics.Precision(
            name="precision"
        ),

        tf.keras.metrics.Recall(
            name="recall"
        ),

        tf.keras.metrics.AUC(
            name="auc"
        )
    ]
)

print("Custom CNN compiled successfully!")

In [ ]:
BEST_MODEL_PATH = (
    MODEL_DIR / "custom_cnn_best.keras"
)

callbacks = [

    # Save the model with the best validation loss
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_MODEL_PATH),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    # Stop training when validation performance stops improving
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),

    # Reduce learning rate when validation loss stagnates
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

In [ ]:
EPOCHS = 10

history = model.fit(

    train_dataset,

    validation_data=val_dataset,

    epochs=EPOCHS,

    callbacks=callbacks,

    verbose=1
)

print("Training completed!")

## Model Evaluation and Artifact Storage

Evaluate the best Custom CNN using validation data.

Save:
- custom_cnn_best.keras
- training_history.csv
- custom_cnn_config.json

Push generated artifacts to GitHub.
The test split remains reserved for final model comparison.

In [ ]:
best_model = tf.keras.models.load_model(
    BEST_MODEL_PATH
)

val_results = best_model.evaluate(
    val_dataset,
    return_dict=True,
    verbose=1
)

print("\nVALIDATION RESULTS")

for metric, value in val_results.items():

    print(
        f"{metric}: {value:.4f}"
    )

In [ ]:
history_df = pd.DataFrame(history.history)

# Save training history
HISTORY_PATH = (
    MODEL_DIR / "training_history.csv"
)

history_df.to_csv(
    HISTORY_PATH,
    index=False
)

# Plot accuracy and loss
fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5)
)

axes[0].plot(
    history_df["accuracy"],
    label="Training Accuracy"
)

axes[0].plot(
    history_df["val_accuracy"],
    label="Validation Accuracy"
)

axes[0].set_title("Training vs Validation Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(alpha=0.3)


axes[1].plot(
    history_df["loss"],
    label="Training Loss"
)

axes[1].plot(
    history_df["val_loss"],
    label="Validation Loss"
)

axes[1].set_title("Training vs Validation Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

display(history_df)

In [ ]:
training_config = {

    "experiment": "Model A - Custom CNN",

    "dataset": DATASET_ID,

    "dataset_revision": HF_REVISION,

    "target": "label_a",

    "class_mapping": {
        "0": "Real",
        "1": "AI-Generated"
    },

    "image_size": IMAGE_SIZE,

    "batch_size": BATCH_SIZE,

    "epochs_requested": EPOCHS,

    "epochs_completed": len(history.epoch),

    "optimizer": "Adam",

    "learning_rate": 0.001,

    "loss": "Binary Crossentropy",

    "class_weights": {
        str(k): float(v)
        for k, v in class_weights.items()
    },

    "seed": SEED,

    "training_samples": len(train_df),

    "validation_samples": len(val_df),

    "model": "custom_cnn_best.keras"
}

CONFIG_PATH = (
    MODEL_DIR / "custom_cnn_config.json"
)

with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        training_config,
        f,
        indent=4
    )

print("Model and experiment configuration saved!")

for path in [
    BEST_MODEL_PATH,
    HISTORY_PATH,
    CONFIG_PATH
]:

    print(
        path.name,
        f"{path.stat().st_size / (1024**2):.2f} MB"
    )

In [ ]:
import base64
from google.colab import userdata

# Retrieve GitHub token from Colab Secrets
token = userdata.get("GITHUB_TOKEN")

if not token:
    raise ValueError("GITHUB_TOKEN is missing!")

# Files to upload
files_to_push = [
    BEST_MODEL_PATH,
    HISTORY_PATH,
    CONFIG_PATH
]

# Verify files and Git ignore status
for file_path in files_to_push:

    if not file_path.exists():
        raise FileNotFoundError(file_path)

    if file_path.stat().st_size >= 100 * 1024 * 1024:
        raise ValueError(
            f"{file_path.name} exceeds GitHub's "
            "100 MB single-file limit. Use Git LFS."
        )

    relative_path = str(
        file_path.relative_to(REPO_DIR)
    )

    ignored = subprocess.run(
        ["git", "check-ignore", "-q", relative_path],
        cwd=REPO_DIR
    )

    if ignored.returncode == 0:
        raise RuntimeError(
            f"{relative_path} is ignored by .gitignore"
        )

# Configure Git identity
subprocess.run(
    [
        "git", "config", "--local",
        "user.name", "Michael Andreas Tjendra"
    ],
    cwd=REPO_DIR,
    check=True
)

subprocess.run(
    [
        "git", "config", "--local",
        "user.email", "alongbest2000@gmail.com"
    ],
    cwd=REPO_DIR,
    check=True
)

# Stage model artifacts
subprocess.run(
    [
        "git", "add",
        "artifacts/models/custom_cnn_best.keras",
        "artifacts/models/training_history.csv",
        "artifacts/models/custom_cnn_config.json"
    ],
    cwd=REPO_DIR,
    check=True
)

# Commit staged files if there are changes
diff_result = subprocess.run(
    ["git", "diff", "--cached", "--quiet"],
    cwd=REPO_DIR
)

if diff_result.returncode != 0:

    subprocess.run(
        [
            "git", "commit",
            "-m", "Train and save Custom CNN model"
        ],
        cwd=REPO_DIR,
        check=True
    )

else:
    print("No new changes to commit.")

# Sync remote before pushing
subprocess.run(
    ["git", "fetch", "origin"],
    cwd=REPO_DIR,
    check=True
)

subprocess.run(
    ["git", "pull", "--rebase", "origin", "main"],
    cwd=REPO_DIR,
    check=True
)

# Temporary Git authentication
askpass_path = Path("/tmp/github-askpass.sh")

askpass_path.write_text("""#!/bin/sh
case "$1" in
    *sername*) printf '%s\\n' 'Michaelatj' ;;
    *assword*) printf '%s\\n' "$GITHUB_TOKEN" ;;
    *) exit 1 ;;
esac
""")

os.chmod(askpass_path, 0o700)

env = os.environ.copy()

env["GIT_ASKPASS"] = str(askpass_path)
env["GIT_TERMINAL_PROMPT"] = "0"
env["GITHUB_TOKEN"] = token

# Push
result = subprocess.run(
    ["git", "push", "origin", "main"],
    cwd=REPO_DIR,
    env=env,
    capture_output=True,
    text=True
)

if result.returncode != 0:

    print(result.stderr)

    raise RuntimeError(
        "GitHub push failed. Check repository synchronization."
    )

print(result.stdout)
print("Custom CNN artifacts successfully pushed!")

In [ ]:
subprocess.run(
    [
        "git",
        "status",
        "--short",
        "--branch"
    ],
    cwd=REPO_DIR,
    check=True
)